In [1]:
!pip install pyotp mplfinance pandas_ta fyers_apiv3 pygame stable-baselines3 gymnasium shimmy optuna higher

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.1/115.1 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.8/171.8 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.1/958.1 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.4/383.4 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 7.9 MB/s eta 0:00:00
 

In [ ]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import mplfinance as mpf
import pandas_ta as ta
import pygame
import os
import pytz
import json
import re
import random
import pickle

import gymnasium as gym
from gymnasium import spaces
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Dropout, Attention, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.callbacks import Callback, ModelCheckpoint
from scipy.signal import argrelextrema
import tensorflow as tf
from collections import deque
from tensorflow import keras
from tensorflow.keras import layers, optimizers, models
from scipy.signal import find_peaks

In [ ]:
def get_index_symbol_and_quantity(index):

    # Dictionary mapping index name to index symbols
    index_symbols = {
        'Bankex': 'BSE:BANKEX-INDEX',
        'Finnifty': 'NSE:FINNIFTY-INDEX',
        'Bank Nifty': 'NSE:NIFTYBANK-INDEX',
        'Nifty': 'NSE:NIFTY50-INDEX',
        'Sensex': 'BSE:SENSEX-INDEX'
    }

    # Determine the index symbol for the given index
    index_symbol = index_symbols.get(index, 'Invalid Index')

    # Determine the quantity based on the index symbol
    if index_symbol == "NSE:NIFTY50-INDEX":
        quantity = 25  # 25 is one lot for Nifty
    elif index_symbol == "NSE:NIFTYBANK-INDEX":
        quantity = 15  # 15 is one lot for Bank Nifty
    elif index_symbol == "NSE:FINNIFTY-INDEX":
        quantity = 40  # 40 is one lot for Finnifty
    elif index_symbol == "BSE:SENSEX-INDEX":
        quantity = 20  # 20 is two lot for Sensex
    elif index_symbol == "BSE:BANKEX-INDEX":
        quantity = 15  # 15 is one lot for Bankex
    else:
        quantity = 0  # Default value if none of the conditions match

    return index_symbol, quantity

In [ ]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = "4628"
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

index_symbol, quantity = get_index_symbol_and_quantity("Bank Nifty")

interval_minutes = 2 # Set the interval to 1, 5, or 15 minutes

ist_timezone = pytz.timezone("Asia/Kolkata")

# Initialize the mixer
#pygame.mixer.init()

# Load the sound file
#pygame.mixer.music.load('sounds/alert.mp3')

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

target = 80
trailing_sl = 40

brokerage = 100

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [ ]:
 session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

,s,code,message,data
fy_id,ok,200,,XM22383
name,ok,200,,MARSHAL TUDU
image,ok,200,,https://myaccount-docs-prod.fyers.in/Profile_P...
display_name,ok,200,,None
pin_change_date,ok,200,,25-09-2023 17:16:16
email_id,ok,200,,iammarshal22@gmail.com
pwd_change_date,ok,200,,01-06-2022 20:36:31
PAN,ok,200,,---------
mobile_number,ok,200,,8458060663
totp,ok,200,,True


In [ ]:
def fetch_candle_data(number):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                train_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                return train_df
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [ ]:
def fetch_train_candle_data(days_count):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [ ]:
#train_df = fetch_candle_data(100)

train_df = fetch_train_candle_data(25)

print(len(train_df))

train_df = train_df.drop_duplicates(subset='datetime', keep='first')

print(len(train_df))

320453
317445


In [ ]:
class FullFeaturePipeline:
    def __init__(self, df: pd.DataFrame, base_interval_minutes: int):
        """
        df: DataFrame with columns [datetime, open, high, low, close, volume (optional)]
            'datetime' is a Unix timestamp in seconds.
        base_interval_minutes: The base timeframe in minutes (e.g. 2, 5, 15, etc.).
        """
        self.df = df.copy()
        self.base_interval = base_interval_minutes

        # We no longer need an htf_map since higher timeframe features are removed
        # self.htf_map = { ... }  # Removed

    def preprocess_datetime(self):
        """
        1) Convert Unix timestamp to local time (Asia/Kolkata).
        2) Check for duplicates/missing timestamps.
        3) Sort by time and set 'datetime' as the index.
        """
        ist = timezone('Asia/Kolkata')

        self.df['datetime'] = pd.to_datetime(self.df['datetime'], unit='s')
        self.df['datetime'] = (self.df['datetime']
                               .dt.tz_localize('UTC')
                               .dt.tz_convert(ist)
                               .dt.tz_localize(None))

        # Check for duplicates/missing
        if self.df['datetime'].duplicated().any() or self.df['datetime'].isnull().any():
            raise ValueError("The 'datetime' column contains duplicates or missing values.")

        # Sort and set index
        self.df.sort_values('datetime', inplace=True)
        self.df.set_index('datetime', inplace=True)
        return self

    def clean_data(self):
        """
        Optionally drop 'volume' column if it contains zeros or NaNs.
        Adjust if you prefer different volume handling.
        """
        if 'volume' in self.df.columns:
            if self.df['volume'].isnull().any() or (self.df['volume'] == 0).any():
                self.df.drop('volume', axis=1, inplace=True, errors='ignore')
        return self

    def add_base_timeframe_features(self):
        """
        Compute a few standard technical indicators on the base timeframe:
        RSI, MACD, Bollinger Bands, ATR, etc.
        """
        self.df.sort_index(inplace=True)

        # RSI on base timeframe close
        self.df['rsi_base'] = ta.rsi(self.df['close'], length=14)

        # MACD
        macd = ta.macd(self.df['close'], fast=12, slow=26, signal=9)
        self.df['macd'] = macd['MACD_12_26_9']
        self.df['macd_h'] = macd['MACDh_12_26_9']
        self.df['macd_s'] = macd['MACDs_12_26_9']

        # Bollinger Bands
        bbands = ta.bbands(self.df['close'], length=20, std=2)
        self.df['bb_lower'] = bbands['BBL_20_2.0']
        self.df['bb_mid'] = bbands['BBM_20_2.0']
        self.df['bb_upper'] = bbands['BBU_20_2.0']

        # ATR (for baseline volatility measure)
        self.df['atr_base'] = ta.atr(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=14
        )

        return self

    def add_candlestick_patterns(self):
        """
        Detect a few common candlestick patterns.
        We store them as numeric columns (0 = not present, 1 = bullish pattern, -1 = bearish pattern, etc.)
        so that the pipeline remains consistent with numeric-only scaling.
        """
        self.df.sort_index(inplace=True)

        # Example: Doji detection (very simplistic)
        # We check if the open and close are nearly the same
        body_size = (self.df['close'] - self.df['open']).abs()
        candle_range = self.df['high'] - self.df['low']
        self.df['candle_doji'] = np.where(
            (body_size <= 0.1 * candle_range), 1.0, 0.0
        )

        # Example: Engulfing pattern (bullish and bearish)
        # - Bullish Engulfing: today's open < yesterday's close, today's close > yesterday's open
        # - Bearish Engulfing: today's open > yesterday's close, today's close < yesterday's open
        # We shift the open/close by 1 day to compare
        self.df['open_shift1'] = self.df['open'].shift(1)
        self.df['close_shift1'] = self.df['close'].shift(1)

        cond_bull_engulf = (self.df['open'] < self.df['close_shift1']) & (self.df['close'] > self.df['open_shift1'])
        cond_bear_engulf = (self.df['open'] > self.df['close_shift1']) & (self.df['close'] < self.df['open_shift1'])

        self.df['candle_engulf'] = np.where(cond_bull_engulf, 1.0,
                                    np.where(cond_bear_engulf, -1.0, 0.0))

        # Drop helper columns to keep dataset clean
        self.df.drop(['open_shift1', 'close_shift1'], axis=1, inplace=True)

        return self

    def add_swing_points(self, left_bars=3, right_bars=3):
        """
        Detect local swing highs and lows.
        left_bars/right_bars define how many bars on each side must be lower/higher (for highs/lows).
        """
        self.df.sort_index(inplace=True)

        # For each row, check if 'high' is greater than the preceding and following N bars
        rolling_high = self.df['high'].rolling(left_bars+1, center=False).max()
        rolling_low = self.df['low'].rolling(left_bars+1, center=False).min()

        # We'll do a simplistic approach: shift rolling computations accordingly
        # A more robust approach might be to iterate row by row, but we'll keep it vectorized for brevity.

        # Swing High
        # Compare current high to future bars as well, so we do a forward rolling
        forward_high = self.df['high'].shift(-right_bars).rolling(right_bars+1, center=False).max()
        self.df['is_swing_high'] = 0.0
        cond_high = (self.df['high'] == rolling_high) & (self.df['high'] == forward_high)
        self.df.loc[cond_high, 'is_swing_high'] = 1.0

        # Swing Low
        forward_low = self.df['low'].shift(-right_bars).rolling(right_bars+1, center=False).min()
        self.df['is_swing_low'] = 0.0
        cond_low = (self.df['low'] == rolling_low) & (self.df['low'] == forward_low)
        self.df.loc[cond_low, 'is_swing_low'] = 1.0

        return self

    def add_range_breakout_features(self, window=20):
        """
        Add Donchian channels, breakout flags, and a simple measure of range expansion.
        """
        self.df.sort_index(inplace=True)

        # Donchian Channels
        self.df['donchian_high'] = self.df['high'].rolling(window).max()
        self.df['donchian_low'] = self.df['low'].rolling(window).min()
        self.df['donchian_range'] = self.df['donchian_high'] - self.df['donchian_low']

        # Breakout flags
        #  1.0 if close > donchian_high, -1.0 if close < donchian_low, else 0.0
        self.df['donchian_breakout'] = np.where(
            self.df['close'] > self.df['donchian_high'], 1.0,
            np.where(self.df['close'] < self.df['donchian_low'], -1.0, 0.0)
        )

        # Range expansion/compression example
        # Rolling std of (high - low)
        self.df['range_expansion'] = (self.df['high'] - self.df['low']).rolling(window).std()

        return self

    def add_momentum_features(self):
        """
        Add additional momentum-based indicators like Stochastics, ADX, CCI, etc.
        (We already have RSI, MACD in add_base_timeframe_features, but you can unify them here if you prefer.)
        """
        self.df.sort_index(inplace=True)

        # Stochastic
        stoch = ta.stoch(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            k=14, d=3
        )
        self.df['stoch_k'] = stoch['STOCHk_14_3_3']
        self.df['stoch_d'] = stoch['STOCHd_14_3_3']

        # ADX (Average Directional Movement Index)
        adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
        self.df['adx'] = adx_data['ADX_14']
        self.df['diplus'] = adx_data['DMP_14']
        self.df['diminus'] = adx_data['DMN_14']

        # CCI (Commodity Channel Index)
        self.df['cci'] = ta.cci(self.df['high'], self.df['low'], self.df['close'], length=20)

        return self

    def add_volatility_features(self, window=20):
        """
        Add additional volatility-based features, e.g. historical volatility, Z-score of returns, etc.
        """
        self.df.sort_index(inplace=True)

        # Historical volatility (using close-to-close log returns)
        self.df['log_return'] = np.log(self.df['close'] / self.df['close'].shift(1))
        self.df['hist_vol'] = self.df['log_return'].rolling(window).std() * np.sqrt(252)  # Annualized approximation

        # Z-Score of Price Changes over 'window'
        rolling_mean = self.df['log_return'].rolling(window).mean()
        rolling_std = self.df['log_return'].rolling(window).std()
        self.df['zscore_return'] = (self.df['log_return'] - rolling_mean) / (rolling_std + 1e-9)

        return self

    def add_volume_features(self):
        """
        Add volume-based features if 'volume' is present.
        """
        if 'volume' not in self.df.columns:
            return self  # No volume data, skip

        self.df.sort_index(inplace=True)

        # On-Balance Volume
        self.df['obv'] = ta.obv(self.df['close'], self.df['volume'])

        # Volume spike detection:
        # Compare current volume to rolling average
        vol_mean = self.df['volume'].rolling(20).mean()
        vol_std = self.df['volume'].rolling(20).std()
        self.df['vol_spike'] = np.where(
            (self.df['volume'] > (vol_mean + 2 * vol_std)), 1.0, 0.0
        )

        # VWAP over the day (if you have daily or session logic, you'd groupby date)
        # Here, for a simpler approach, we do a rolling 1-day approximation:
        # This might not be perfectly accurate if you have continuous 24h data, but it’s a start.
        # Usually you’d reset VWAP each day or each session.
        typical_price = (self.df['high'] + self.df['low'] + self.df['close']) / 3.0
        self.df['cum_tp_vol'] = (typical_price * self.df['volume']).cumsum()
        self.df['cum_vol'] = self.df['volume'].cumsum()
        self.df['vwap'] = self.df['cum_tp_vol'] / (self.df['cum_vol'] + 1e-9)

        # Drop helper columns
        self.df.drop(['cum_tp_vol', 'cum_vol'], axis=1, inplace=True)

        return self

    def add_regime_features(self):
        """
        Classify each bar as 'trending' or 'ranging' (0 or 1) based on ADX, or volatility-based thresholds, etc.
        Keep it numeric for scaling (e.g., 0.0 or 1.0).
        """
        self.df.sort_index(inplace=True)

        # If ADX above 25 => trending, else ranging
        # (25 is just a typical reference threshold; pick whichever fits your strategy)
        if 'adx' not in self.df.columns:
            # If ADX wasn't yet computed, do it quickly
            adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
            self.df['adx'] = adx_data['ADX_14']

        self.df['regime_trend'] = np.where(self.df['adx'] >= 25, 1.0, 0.0)

        # Alternatively, you could have multiple columns for different regime classifications
        return self

    def add_time_features(self):
        """
        Extract cyclical or numeric time features (hour of day, day of week).
        We ensure they are numeric (float).
        """
        self.df.sort_index(inplace=True)

        # Because 'datetime' is now our index, we can access it via self.df.index
        # Create columns for hour, day of week
        self.df['hour'] = self.df.index.hour.astype(float)
        self.df['day_of_week'] = self.df.index.dayofweek.astype(float)

        # Cyclical encoding for hour
        self.df['hour_sin'] = np.sin(2 * np.pi * self.df['hour'] / 24.0)
        self.df['hour_cos'] = np.cos(2 * np.pi * self.df['hour'] / 24.0)

        # Cyclical encoding for day_of_week
        self.df['dow_sin'] = np.sin(2 * np.pi * self.df['day_of_week'] / 7.0)
        self.df['dow_cos'] = np.cos(2 * np.pi * self.df['day_of_week'] / 7.0)

        # You can drop raw hour/day_of_week if you prefer only cyclical features
        # self.df.drop(['hour', 'day_of_week'], axis=1, inplace=True)

        return self

    def add_adaptive_targets_and_stops(self):
        """
        Instead of fixed (2 * ATR) / (1 * ATR), adapt based on current volatility or regime.
        For demonstration, we’ll say:
            - If regime is trending, target = 3 * ATR, stop = 1.5 * ATR
            - If regime is ranging, target = 1.5 * ATR, stop = 1 * ATR
        This is just an example logic.
        """
        self.df.sort_index(inplace=True)

        if 'atr_base' not in self.df.columns:
            self.df['atr_base'] = ta.atr(self.df['high'], self.df['low'], self.df['close'], length=14)

        # If we haven't computed 'regime_trend', do so
        if 'regime_trend' not in self.df.columns:
            self.add_regime_features()

        # Example adaptation
        is_trend = (self.df['regime_trend'] == 1.0)
        self.df['Target'] = np.where(is_trend, 3.0 * self.df['atr_base'], 1.5 * self.df['atr_base'])
        self.df['StopLoss'] = np.where(is_trend, 1.5 * self.df['atr_base'], 1.0 * self.df['atr_base'])

        return self

    def run_pipeline(self):
        """
        Run all steps except higher timeframe computations (now removed).
        You can rearrange the order as desired.
        """
        (self.preprocess_datetime()
             .clean_data()
             .add_base_timeframe_features()
             .add_candlestick_patterns()
             .add_swing_points()
             .add_range_breakout_features()
             .add_momentum_features()
             .add_volatility_features()
             .add_volume_features()            # only applies if 'volume' exists
             .add_regime_features()
             .add_time_features()
             .add_adaptive_targets_and_stops()
        )
        return self

    def get_processed_df(self):
        """
        Retrieve the final DataFrame (including any signals if label_signals() was called).
        We'll also ensure that all columns except 'Signal' are float with 2 decimal places.
        """
        # Drop rows that have NaN from lookbacks
        self.df.dropna(axis=0, how='any', inplace=True)

        # Drop 'Entry Price' and 'Exit Price' if you do not want them in final features
        if 'Entry Price' in self.df.columns:
            self.df.drop('Entry Price', axis=1, inplace=True)
        if 'Exit Price' in self.df.columns:
            self.df.drop('Exit Price', axis=1, inplace=True)

        # Ensure 'Signal' is int; everything else float with 2 decimals
        if 'Signal' in self.df.columns:
            sig_col = self.df['Signal'].astype(int)
            self.df.drop(columns=['Signal'], inplace=True)
        else:
            sig_col = None

        # Convert all remaining columns to float
        for col in self.df.columns:
            self.df[col] = self.df[col].astype(float)

        # Round to 2 decimals
        self.df = self.df.round(2)

        # Reinsert Signal column if it exists
        if sig_col is not None:
            self.df['Signal'] = sig_col

        return self.df

train_pipeline = FullFeaturePipeline(train_df, interval_minutes)
train_pipeline.run_pipeline()

processed_train_df = train_pipeline.get_processed_df()
processed_train_df

,open,high,low,close,rsi_base,macd,macd_h,macd_s,bb_lower,bb_mid,...,zscore_return,regime_trend,hour,day_of_week,hour_sin,hour_cos,dow_sin,dow_cos,Target,StopLoss
datetime,,,,,,,,,,,,,,,,,,,,,
2017-12-11 10:21:00,25405.20,25405.20,25395.50,25403.30,33.35,-3.38,-3.59,0.21,25401.65,25420.66,...,0.06,0.0,10.0,0.0,0.50,-0.87,0.00,1.00,17.79,11.86
2017-12-11 10:23:00,25402.10,25418.60,25402.00,25417.20,49.12,-2.85,-2.45,-0.40,25401.31,25420.25,...,2.28,0.0,10.0,0.0,0.50,-0.87,0.00,1.00,18.34,12.23
2017-12-11 10:25:00,25416.50,25425.20,25414.20,25415.70,47.80,-2.52,-1.69,-0.82,25400.88,25419.78,...,-0.16,0.0,10.0,0.0,0.50,-0.87,0.00,1.00,18.20,12.13
2017-12-11 10:27:00,25415.90,25421.30,25411.10,25415.30,47.44,-2.26,-1.15,-1.11,25400.60,25419.60,...,-0.04,0.0,10.0,0.0,0.50,-0.87,0.00,1.00,17.98,11.99
2017-12-11 10:29:00,25415.90,25418.20,25408.30,25415.10,47.24,-2.05,-0.75,-1.30,25400.23,25419.32,...,0.01,0.0,10.0,0.0,0.50,-0.87,0.00,1.00,17.74,11.83
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-10-15 15:21:00,51907.25,51936.00,51905.85,51936.00,59.03,11.11,6.96,4.14,51824.41,51883.28,...,0.85,0.0,15.0,1.0,-0.71,-0.71,0.78,0.62,57.43,38.29
2024-10-15 15:23:00,51935.85,51955.75,51932.80,51950.15,61.10,14.32,8.14,6.18,51826.39,51888.73,...,0.32,0.0,15.0,1.0,-0.71,-0.71,0.78,0.62,55.78,37.19
2024-10-15 15:25:00,51950.80,51984.15,51950.10,51978.35,64.92,18.92,10.19,8.73,51820.56,51893.80,...,0.86,0.0,15.0,1.0,-0.71,-0.71,0.78,0.62,55.45,36.97


In [ ]:
train_pipeline.get_processed_df().columns

Index(['open', 'high', 'low', 'close', 'rsi_base', 'macd', 'macd_h', 'macd_s',
       'bb_lower', 'bb_mid', 'bb_upper', 'atr_base', 'candle_doji',
       'candle_engulf', 'is_swing_high', 'is_swing_low', 'donchian_high',
       'donchian_low', 'donchian_range', 'donchian_breakout',
       'range_expansion', 'stoch_k', 'stoch_d', 'adx', 'diplus', 'diminus',
       'cci', 'log_return', 'hist_vol', 'zscore_return', 'regime_trend',
       'hour', 'day_of_week', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
       'Target', 'StopLoss'],
      dtype='object')

MAML RL Model

In [3]:
################################################################################
# MAML + Transformer for Discrete Action Trading (Fixed & Enhanced)
#
# This is a fully functional Jupyter Notebook in Python 3.x that demonstrates
# an advanced MAML approach with a Transformer-based policy to learn discrete
# trading actions [HOLD, BUY, SELL, CLOSE]. It addresses the following:
#
#   1) Ensuring the model does not hold a position for too many bars/candles
#      by adding a configurable penalty for each step the agent stays in a
#      position, demotivating endless holding.
#   2) Only one position can be active at a time. The agent must choose
#      action CLOSE to exit, and only then can it BUY or SELL again.
#   3) MAML is used so the model can meta-learn across multiple tasks/slices
#      of data, then adapt quickly to new data.
#   4) The code fixes potential NaN issues in the policy outputs by gradient
#      clipping and numeric clamping (nan_to_num).
#   5) The final performance is judged by final capital and win rate. We
#      include a more direct penalty for holding positions too long, so that
#      the policy learns to close promptly.
#
# Instructions:
#   1) In a terminal or a notebook cell, install dependencies if missing:
#         pip install torch higher numpy pandas matplotlib
#   2) Ensure you have a pandas DataFrame named "processed_train_df" in your
#      environment. It must have a 'close' column and any other feature columns
#      with no NaNs. For safety:
#           processed_train_df.dropna(inplace=True)
#      The code references "processed_train_df" below.
#   3) Run all cells in order.
#
# The final model parameters are saved to "maml_trader_policy.pt".
################################################################################

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import matplotlib.pyplot as plt
from typing import List, Tuple, Optional
import random
import os

# For second-order meta-learning (MAML)
import higher

# Ensure plots appear in-line in Jupyter
%matplotlib inline

################################################################################
# 0. Configuration Dictionary
################################################################################

config = {
    # Environment hyperparams
    "seq_len": 16,
    "initial_capital": 100000.0,
    "brokerage": 20.0,
    "quantity": 50,
    "repeated_action_penalty": 0.05,
    "drawdown_penalty": 0.1,
    "margin_call": 0.3,        # 30% of initial capital
    "reward_scale": 1e-3,      # Scales each step reward (PnL) by 1e-3
    "holding_penalty": 0.02,   # Negative reward each step a position is held

    # MAML hyperparams
    "lr_inner": 1e-3,
    "lr_meta": 1e-4,
    "gamma": 0.99,
    "inner_steps": 2,
    "max_steps_per_env": 200,
    "weight_decay": 1e-5,

    # Gradient clipping
    "grad_clip": 5.0,         # Clip gradient norm to 5.0

    # Training tasks
    "num_tasks": 3,
    "task_size": 1000,
    "num_meta_iterations": 5,
    "shuffle_tasks": True,

    # Model save path
    "model_file": "maml_trader_policy.pt"
}

################################################################################
# 1. Detect GPU and Configure Device
################################################################################

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

################################################################################
# 2. Environment with 4 Actions (Hold, Buy, Sell, Close)
#    - Includes a penalty for staying too long in a position (holding_penalty).
################################################################################

class GPUTradingEnv:
    """
    A trading environment with discrete actions:
        0 = Hold
        1 = Buy
        2 = Sell
        3 = Close

    * Only one position at a time (either long or short). The agent must CLOSE
      before opening a new position.
    * Includes repeated-action penalty, drawdown penalty, margin call, and an
      extra penalty for holding a position too many steps (holding_penalty).
    * A small reward_scale factor is applied to the PnL so it doesn't blow up
      the gradients.
    """

    def __init__(
        self,
        df: pd.DataFrame,
        seq_len: int,
        initial_capital: float,
        brokerage: float,
        quantity: int,
        repeated_action_penalty: float,
        drawdown_penalty: float,
        margin_call: float,
        reward_scale: float,
        holding_penalty: float
    ):
        """
        :param df: DataFrame with columns including 'close' + features. No NaNs.
        :param seq_len: Number of timesteps in each observation.
        :param initial_capital: Starting capital.
        :param brokerage: Flat fee for each transaction (entry/exit).
        :param quantity: Number of shares/contracts per position.
        :param repeated_action_penalty: Penalty for repeating same action 4+ times.
        :param drawdown_penalty: Penalty each step if capital < initial_capital.
        :param margin_call: If capital < margin_call * initial_capital, forced close.
        :param reward_scale: Scale factor for realized PnL.
        :param holding_penalty: Negative reward each step in an open position.
        """
        self.df = df.copy()
        self.df.reset_index(drop=True, inplace=True)

        self.seq_len = seq_len
        self.initial_capital = initial_capital
        self.brokerage = brokerage
        self.quantity = quantity
        self.repeated_action_penalty = repeated_action_penalty
        self.drawdown_penalty = drawdown_penalty
        self.margin_call = margin_call
        self.reward_scale = reward_scale
        self.holding_penalty = holding_penalty

        self.current_step = 0
        self.done = False
        self.capital = initial_capital
        self.position = None  # 'long' or 'short'
        self.entry_price = 0.0

        self.time_in_position = 0  # how many steps we have held current position
        self.last_action = None
        self.repeat_count = 0

        self.trade_log = []
        self.max_step = len(self.df) - 1

    def _get_observation(self) -> np.ndarray:
        """
        Returns the last `seq_len` rows as a numpy array,
        shape (seq_len, num_features). If not enough history, pad with zeros.
        """
        start_idx = max(0, self.current_step - self.seq_len + 1)
        obs_data = self.df.iloc[start_idx:self.current_step + 1].values
        if len(obs_data) < self.seq_len:
            pad_len = self.seq_len - len(obs_data)
            obs_data = np.concatenate(
                [np.zeros((pad_len, obs_data.shape[1])), obs_data],
                axis=0
            )
        return obs_data

    def reset(self):
        self.current_step = 0
        self.done = False
        self.capital = self.initial_capital
        self.position = None
        self.entry_price = 0.0
        self.time_in_position = 0
        self.last_action = None
        self.repeat_count = 0
        self.trade_log = []
        return self._get_observation()

    def step(self, action: int):
        """
        :param action: one of {0=Hold, 1=Buy, 2=Sell, 3=Close}
        :return: (next_obs, reward, done, info)
        """
        if self.done:
            return self._get_observation(), 0.0, True, {}

        row = self.df.iloc[self.current_step]
        current_price = row['close']
        reward = 0.0
        info = {}

        # Check repeated action
        if self.last_action == action:
            self.repeat_count += 1
        else:
            self.repeat_count = 1
        self.last_action = action

        # Margin call check (if in a position)
        if self.position is not None and self.capital < self.margin_call * self.initial_capital:
            # Forced close
            forced_pnl = 0.0
            if self.position == 'long':
                forced_pnl = (current_price - self.entry_price) * self.quantity
            else:  # short
                forced_pnl = (self.entry_price - current_price) * self.quantity

            self.capital += forced_pnl
            self.capital -= self.brokerage

            reward += forced_pnl * self.reward_scale  # scaled PnL

            self.trade_log.append({
                'Step': self.current_step,
                'Position': self.position,
                'EntryPrice': self.entry_price,
                'ExitPrice': current_price,
                'PnL': forced_pnl,
                'Capital': self.capital,
                'ForcedClose': True
            })

            # Reset position
            self.position = None
            self.entry_price = 0.0
            self.time_in_position = 0

        # Normal action logic:
        #  1) BUY or SELL only if no position is open
        #  2) CLOSE only if a position is open
        #  3) Otherwise HOLD
        if action == 1 and self.position is None:  # Buy
            self.position = 'long'
            self.entry_price = current_price
            self.capital -= self.brokerage
            self.time_in_position = 0

        elif action == 2 and self.position is None:  # Sell
            self.position = 'short'
            self.entry_price = current_price
            self.capital -= self.brokerage
            self.time_in_position = 0

        elif action == 3 and self.position is not None:  # Close
            if self.position == 'long':
                pnl = (current_price - self.entry_price) * self.quantity
            else:  # 'short'
                pnl = (self.entry_price - current_price) * self.quantity

            self.capital += pnl
            self.capital -= self.brokerage

            reward += pnl * self.reward_scale  # scaled PnL

            self.trade_log.append({
                'Step': self.current_step,
                'Position': self.position,
                'EntryPrice': self.entry_price,
                'ExitPrice': current_price,
                'PnL': pnl,
                'Capital': self.capital,
                'ForcedClose': False
            })

            # Reset position
            self.position = None
            self.entry_price = 0.0
            self.time_in_position = 0

        # If we are still in a position, increment time_in_position
        if self.position is not None:
            self.time_in_position += 1
            # Small negative reward for each bar we hold a position
            reward -= self.holding_penalty

        # Additional penalty if capital < initial
        if self.capital < self.initial_capital:
            reward -= self.drawdown_penalty

        # Penalty if repeating the same action too many times
        if self.repeat_count >= 4:
            reward -= self.repeated_action_penalty

        # Advance step
        self.current_step += 1
        if self.current_step >= self.max_step:
            # End of data => done
            self.done = True
            # Force close if still in a position
            if self.position is not None:
                final_price = self.df.iloc[self.current_step - 1]['close']
                if self.position == 'long':
                    pnl = (final_price - self.entry_price) * self.quantity
                else:
                    pnl = (self.entry_price - final_price) * self.quantity
                self.capital += pnl
                self.capital -= self.brokerage
                reward += pnl * self.reward_scale

                self.trade_log.append({
                    'Step': self.current_step,
                    'Position': self.position,
                    'EntryPrice': self.entry_price,
                    'ExitPrice': final_price,
                    'PnL': pnl,
                    'Capital': self.capital,
                    'ForcedClose': True
                })
                self.position = None
                self.entry_price = 0.0
                self.time_in_position = 0

        next_obs = self._get_observation()
        return next_obs, reward, self.done, info

    def get_trade_log(self) -> pd.DataFrame:
        return pd.DataFrame(self.trade_log)

    def get_final_capital(self) -> float:
        return self.capital

################################################################################
# 3. Deeper Transformer Policy (batch_first=True), GPU-Compatible
################################################################################

class DeeperTransformerPolicy(nn.Module):
    """
    A multi-layer Transformer-based policy for action logits:
    Actions: 0=Hold, 1=Buy, 2=Sell, 3=Close.
    """

    def __init__(
        self,
        num_features: int,
        num_actions: int = 4,
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 3,
        dim_feedforward: int = 256,
        dropout: float = 0.2
    ):
        super().__init__()

        self.num_features = num_features
        self.d_model = d_model

        # Linear embedding to project features -> d_model
        self.embedding = nn.Linear(num_features, d_model)

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='relu',
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # MLP head to produce logits
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_actions)
        )

    def forward(self, x: torch.Tensor):
        """
        :param x: shape (batch_size, seq_len, num_features)
        :return: logits, shape (batch_size, num_actions)
        """
        # Project features
        x_emb = self.embedding(x)  # -> (batch_size, seq_len, d_model)

        # Pass through the Transformer
        x_enc = self.transformer_encoder(x_emb)  # -> (batch_size, seq_len, d_model)

        # Take the last time-step
        x_final = x_enc[:, -1, :]  # -> (batch_size, d_model)

        # Feedforward head for action logits
        logits = self.head(x_final)  # -> (batch_size, num_actions)
        return logits

################################################################################
# 4. Advanced MAML Trader with Gradient Clipping & GPU Support
################################################################################

class AdvancedMAMLTrader:
    """
    Uses second-order MAML (via 'higher') for RL tasks.
    Incorporates gradient clipping to avoid producing NaN logits.
    """

    def __init__(self, policy: nn.Module, cfg: dict):
        """
        :param policy: The base policy network (must be on GPU).
        :param cfg: Dictionary of hyperparameters from 'config'.
        """
        self.device = device
        self.policy = policy.to(self.device)

        self.lr_inner = cfg["lr_inner"]
        self.lr_meta = cfg["lr_meta"]
        self.gamma = cfg["gamma"]
        self.inner_steps = cfg["inner_steps"]
        self.max_steps_per_env = cfg["max_steps_per_env"]
        self.weight_decay = cfg["weight_decay"]
        self.grad_clip = cfg["grad_clip"]

        self.meta_optimizer = optim.Adam(
            self.policy.parameters(),
            lr=self.lr_meta,
            weight_decay=self.weight_decay
        )

    def collect_episodes(self, env: GPUTradingEnv, model: nn.Module, max_steps=None):
        """
        Run a single rollout in the environment using 'model'.
        Returns a list of (obs, action, reward).
        """
        if max_steps is None:
            max_steps = self.max_steps_per_env

        transitions = []
        obs = env.reset()
        done = False
        step_count = 0

        while not done and step_count < max_steps:
            obs_np = np.array([obs], dtype=np.float32)  # (1, seq_len, num_features)
            obs_t = torch.from_numpy(obs_np).to(self.device)

            with torch.no_grad():
                logits = model(obs_t)
                # numeric clamp
                logits = torch.nan_to_num(logits, nan=0.0, posinf=1e3, neginf=-1e3)

                probs = F.softmax(logits, dim=-1)
                probs = torch.nan_to_num(probs, nan=1e-8)
                probs = probs / probs.sum(dim=-1, keepdim=True)  # re-normalize

                probs_cpu = probs[0].cpu().numpy()
                # Stochastic sampling for exploration
                action = np.random.choice(len(probs_cpu), p=probs_cpu)

            next_obs, reward, done, _ = env.step(action)
            transitions.append((obs, action, reward))

            obs = next_obs
            step_count += 1

        return transitions

    def compute_returns(self, rewards: List[float]) -> torch.Tensor:
        """
        Compute discounted returns from a list of rewards.
        Returns a 1D torch.Tensor on GPU.
        """
        R = 0.0
        returns = []
        for r in reversed(rewards):
            R = r + self.gamma * R
            returns.insert(0, R)
        return torch.tensor(returns, dtype=torch.float32, device=self.device)

    def inner_update(self, env: GPUTradingEnv, fmodel, diffopt):
        """
        Perform `self.inner_steps` gradient updates on the cloned model (fmodel)
        for the given environment.
        """
        for _ in range(self.inner_steps):
            transitions = self.collect_episodes(env, fmodel)
            if len(transitions) == 0:
                continue

            obs_list, action_list, reward_list = [], [], []
            for (obs, act, rew) in transitions:
                obs_list.append(obs)
                action_list.append(act)
                reward_list.append(rew)

            if len(obs_list) == 0:
                continue

            obs_np = np.array(obs_list, dtype=np.float32)
            obs_t = torch.from_numpy(obs_np).to(self.device)
            returns_t = self.compute_returns(reward_list)

            logits = fmodel(obs_t)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=1e3, neginf=-1e3)

            log_probs = F.log_softmax(logits, dim=-1)
            chosen_log_probs = log_probs[range(len(action_list)), action_list]

            # Policy gradient loss
            loss = - (returns_t * chosen_log_probs).mean()

            # second-order "higher" approach
            diffopt.step(loss)

        return fmodel

    def meta_update(self, envs: List[GPUTradingEnv]):
        """
        Meta-update over multiple tasks (envs).
         1) For each env, clone the policy -> do inner updates -> gather new
            rollout -> compute meta-loss vs. the base policy.
         2) Sum or average the meta-loss, then step the meta-optimizer.
        """
        meta_loss = 0.0
        task_count = len(envs)
        self.meta_optimizer.zero_grad()

        for env in envs:
            # Fresh start
            env.reset()

            # Inner loop optimizer
            inner_opt = torch.optim.SGD(self.policy.parameters(), lr=self.lr_inner)

            with higher.innerloop_ctx(self.policy, inner_opt, copy_initial_weights=True) as (fmodel, diffopt):
                # 1) Inner adaptation
                self.inner_update(env, fmodel, diffopt)

                # 2) Gather transitions for meta-loss
                transitions = self.collect_episodes(env, fmodel)
                if len(transitions) == 0:
                    continue

                obs_list, action_list, reward_list = [], [], []
                for (obs, act, rew) in transitions:
                    obs_list.append(obs)
                    action_list.append(act)
                    reward_list.append(rew)

                if len(obs_list) == 0:
                    continue

                obs_np = np.array(obs_list, dtype=np.float32)
                obs_t = torch.from_numpy(obs_np).to(self.device)
                returns_t = self.compute_returns(reward_list)

                # Evaluate base policy (not fmodel) for the meta-loss
                base_logits = self.policy(obs_t)
                base_logits = torch.nan_to_num(base_logits, nan=0.0, posinf=1e3, neginf=-1e3)

                base_log_probs = F.log_softmax(base_logits, dim=-1)
                chosen_log_probs = base_log_probs[range(len(action_list)), action_list]

                task_loss = - (returns_t * chosen_log_probs).mean()
                meta_loss += task_loss

        if task_count > 0:
            meta_loss = meta_loss / task_count
            meta_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.policy.parameters(), max_norm=self.grad_clip)
            self.meta_optimizer.step()

        return meta_loss.item() if task_count > 0 else 0.0

    def online_adaptation(self, new_env: GPUTradingEnv, adaptation_steps=1):
        """
        Adapt the current policy to new data using a few gradient steps
        directly on the existing policy.
        """
        inner_opt = torch.optim.SGD(self.policy.parameters(), lr=self.lr_inner)

        for _ in range(adaptation_steps):
            transitions = self.collect_episodes(new_env, self.policy)
            if len(transitions) == 0:
                continue

            obs_list, action_list, reward_list = [], [], []
            for (obs, act, rew) in transitions:
                obs_list.append(obs)
                action_list.append(act)
                reward_list.append(rew)

            if len(obs_list) == 0:
                continue

            obs_np = np.array(obs_list, dtype=np.float32)
            obs_t = torch.from_numpy(obs_np).to(self.device)
            returns_t = self.compute_returns(reward_list)

            logits = self.policy(obs_t)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=1e3, neginf=-1e3)

            log_probs = F.log_softmax(logits, dim=-1)
            chosen_log_probs = log_probs[range(len(action_list)), action_list]

            loss = - (returns_t * chosen_log_probs).mean()

            inner_opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.policy.parameters(), max_norm=self.grad_clip)
            inner_opt.step()

################################################################################
# 5. Train & Evaluate Functions
################################################################################

def train_maml_on_multiple_tasks(df: pd.DataFrame, cfg: dict):
    """
    1) Create multiple environment tasks from df by sampling random slices
       of length cfg["task_size"].
    2) Run meta-updates for cfg["num_meta_iterations"].
    3) Save the trained policy to cfg["model_file"].

    If cfg["model_file"] exists, it is loaded first (to continue training).
    Returns (policy, envs, maml).
    """
    task_size = cfg["task_size"]
    seq_len = cfg["seq_len"]
    num_tasks = cfg["num_tasks"]
    shuffle_tasks = cfg["shuffle_tasks"]
    num_meta_iterations = cfg["num_meta_iterations"]
    model_file = cfg["model_file"]

    initial_capital = cfg["initial_capital"]
    brokerage = cfg["brokerage"]
    quantity = cfg["quantity"]
    repeated_action_penalty = cfg["repeated_action_penalty"]
    drawdown_penalty = cfg["drawdown_penalty"]
    margin_call = cfg["margin_call"]
    reward_scale = cfg["reward_scale"]
    holding_penalty = cfg["holding_penalty"]

    if len(df) < task_size:
        raise ValueError(f"DataFrame must have at least {task_size} rows, found {len(df)}.")
    if 'close' not in df.columns:
        raise ValueError("DataFrame must contain a 'close' column.")

    # Ensure no NaNs remain (safety check)
    if df.isna().any().any():
        raise ValueError("DataFrame still contains NaN values. Please clean them before training.")

    num_features = df.shape[1]

    # Create or load a policy
    policy = DeeperTransformerPolicy(num_features=num_features, num_actions=4).to(device)
    if os.path.exists(model_file):
        print(f"Loading existing policy from '{model_file}' ...")
        policy.load_state_dict(torch.load(model_file, map_location=device))
    else:
        print("No existing policy found. Creating a fresh policy.")

    trader = AdvancedMAMLTrader(policy=policy, cfg=cfg)

    # Create tasks
    envs = []
    indices = []

    if shuffle_tasks:
        for _ in range(num_tasks):
            start = random.randint(0, len(df) - task_size)
            end = start + task_size
            indices.append((start, end))
    else:
        chunk_size = len(df) // num_tasks
        for i in range(num_tasks):
            start = i * chunk_size
            end = start + task_size
            if end > len(df):
                end = len(df)
            indices.append((start, end))

    for (st, ed) in indices:
        slice_df = df.iloc[st:ed].copy()
        env = GPUTradingEnv(
            df=slice_df,
            seq_len=seq_len,
            initial_capital=initial_capital,
            brokerage=brokerage,
            quantity=quantity,
            repeated_action_penalty=repeated_action_penalty,
            drawdown_penalty=drawdown_penalty,
            margin_call=margin_call,
            reward_scale=reward_scale,
            holding_penalty=holding_penalty
        )
        envs.append(env)

    # Meta-training loop
    for iteration in range(num_meta_iterations):
        meta_loss = trader.meta_update(envs)
        print(f"Iteration {iteration+1}/{num_meta_iterations}, Meta-Loss: {meta_loss:.6f}")

    # Save updated policy
    torch.save(trader.policy.state_dict(), model_file)
    print(f"Policy saved to '{model_file}'")

    return trader.policy, envs, trader


def evaluate_policy(env: GPUTradingEnv, policy: nn.Module, max_steps=200):
    """
    Evaluate a given policy on a single environment.
    Returns a dict with final capital, trade log, and stats.
    """
    policy.eval()
    obs = env.reset()
    done = False
    step = 0

    with torch.no_grad():
        while not done and step < max_steps:
            obs_np = np.array([obs], dtype=np.float32)  # (1, seq_len, num_features)
            obs_t = torch.from_numpy(obs_np).to(device)

            logits = policy(obs_t)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=1e3, neginf=-1e3)

            probs = F.softmax(logits, dim=-1)
            probs = torch.nan_to_num(probs, nan=1e-8)
            probs = probs / probs.sum(dim=-1, keepdim=True)

            # Greedy in evaluation
            action = probs[0].argmax().cpu().item()

            obs, _, done, _ = env.step(action)
            step += 1

    final_cap = env.get_final_capital()
    trades = env.get_trade_log()

    if not trades.empty:
        wins = sum(trades['PnL'] > 0)
        total_trades = len(trades)
        win_rate = float(wins) / total_trades if total_trades > 0 else 0.0
    else:
        win_rate = 0.0

    return {
        'FinalCapital': final_cap,
        'TradeLog': trades,
        'WinRate': win_rate
    }


################################################################################
# 6. MAIN - Example Usage
################################################################################

if __name__ == "__main__":
    try:
        print("Starting MAML training...")

        # You must supply a "processed_train_df" with at least:
        #   - A 'close' column
        #   - No NaNs
        #   - Enough rows to cover config["task_size"]
        #
        # Example (pseudo):
        # processed_train_df = your_dataframe_with_features
        # processed_train_df.dropna(inplace=True)
        # print("DataFrame size:", processed_train_df.shape)
        #
        # Then pass to train_maml_on_multiple_tasks:

        trained_policy, envs, maml_trader = train_maml_on_multiple_tasks(
            df=processed_train_df,
            cfg=config
        )

        print("\n--- MAML Training Complete ---")

        # Evaluate on one of the trained tasks
        test_env = envs[0]
        eval_results = evaluate_policy(test_env, trained_policy, max_steps=200)
        print("Evaluation Final Capital:", eval_results['FinalCapital'])
        print("Evaluation Win Rate:", eval_results['WinRate'])
        print("Trade Log (head):\n", eval_results['TradeLog'].head())

    except NameError:
        print("ERROR: 'processed_train_df' not found. Please define it before running.")
    except Exception as e:
        print("Exception during MAML training.")

ModuleNotFoundError: No module named 'higher'

In [ ]:
def get_sleep_time(interval_minutes, market_start_hour=9, market_start_minute=15, market_close_hour=15, market_close_minute=0):
    # Get current time in IST
    now = datetime.now(pytz.utc).astimezone(ist_timezone)

    # Define the market start and close times in IST for today
    market_start_time = now.replace(hour=market_start_hour, minute=market_start_minute, second=0, microsecond=0)
    market_close_time = now.replace(hour=market_close_hour, minute=market_close_minute, second=0, microsecond=0)

    if now < market_start_time:
        # If current time is before market starts, set next_run_time to market start time
        next_run_time = market_start_time
    elif now > market_close_time:
        # If current time is after market close, calculate the time until the next market open
        next_market_start_time = market_start_time + timedelta(days=1)
        next_run_time = next_market_start_time
    else:
        # Calculate the minutes since the market start time
        minutes_since_market_start = (now - market_start_time).total_seconds() // 60
        # Calculate the number of minutes to the next interval boundary
        minutes_to_next_interval = interval_minutes - (minutes_since_market_start % interval_minutes)
        # Calculate the next run time by adding these minutes to the current time
        next_run_time = (now + timedelta(minutes=minutes_to_next_interval)).replace(second=0, microsecond=0)

    # Calculate the sleep time in seconds
    sleep_time = (next_run_time - now).total_seconds()
    return sleep_time

In [ ]:
def fetch_option_chain():
    while True:
        try:
            data = {
                "symbol": index_symbol,
                "strikecount": 2,
                "timestamp": ""
            }
            response = fyers.optionchain(data=data)

            if response is not None:
                return response
        except Exception as e:
            print(f"Error fetching Option Chain: {e}")
            time.sleep(active_order_sleep)

index_oc= fetch_option_chain()

pd.DataFrame(index_oc['data']['optionsChain'])

,ask,bid,description,ex_symbol,exchange,fp,fpch,fpchp,fyToken,ltp,ltpch,ltpchp,option_type,strike_price,symbol,oi,oich,oichp,prev_oi,volume
0,0.00,0.00,NIFTYBANK-INDEX,BANKNIFTY,NSE,48871.0,107.75,0.22,101000000026009,48724.40,153.50,0.32,,-1,NSE:NIFTYBANK-INDEX,NaN,NaN,NaN,NaN,NaN
1,363.40,360.00,NaN,NaN,NaN,NaN,NaN,NaN,101125013039478,363.10,-138.95,-27.68,PE,48500,NSE:BANKNIFTY25JAN48500PE,987990.0,110445.0,12.59,877545.0,18391650.0
2,739.00,738.05,NaN,NaN,NaN,NaN,NaN,NaN,101125013039477,739.00,-25.10,-3.28,CE,48500,NSE:BANKNIFTY25JAN48500CE,729300.0,57585.0,8.57,671715.0,14100555.0
3,401.20,400.00,NaN,NaN,NaN,NaN,NaN,NaN,101125013039480,400.00,-142.60,-26.28,PE,48600,NSE:BANKNIFTY25JAN48600PE,373110.0,120105.0,47.47,253005.0,11321670.0
4,679.80,675.00,NaN,NaN,NaN,NaN,NaN,NaN,101125013039479,679.95,-30.55,-4.30,CE,48600,NSE:BANKNIFTY25JAN48600CE,370020.0,64080.0,20.95,305940.0,10199115.0
5,443.20,441.15,NaN,NaN,NaN,NaN,NaN,NaN,101125013039482,441.00,-146.00,-24.87,PE,48700,NSE:BANKNIFTY25JAN48700PE,303675.0,63360.0,26.37,240315.0,8751120.0
6,622.00,619.70,NaN,NaN,NaN,NaN,NaN,NaN,101125013039481,619.70,-39.40,-5.98,CE,48700,NSE:BANKNIFTY25JAN48700CE,450195.0,150090.0,50.01,300105.0,10277115.0
7,486.85,485.75,NaN,NaN,NaN,NaN,NaN,NaN,101125013039485,483.85,-146.25,-23.21,PE,48800,NSE:BANKNIFTY25JAN48800PE,249060.0,-48345.0,-16.26,297405.0,4721700.0
8,565.00,563.10,NaN,NaN,NaN,NaN,NaN,NaN,101125013039483,565.00,-44.90,-7.36,CE,48800,NSE:BANKNIFTY25JAN48800CE,465435.0,8655.0,1.89,456780.0,6651360.0
9,538.95,534.90,NaN,NaN,NaN,NaN,NaN,NaN,101125013039487,538.10,-147.70,-21.54,PE,48900,NSE:BANKNIFTY25JAN48900PE,180450.0,-40065.0,-18.17,220515.0,2446980.0


In [ ]:
def assign_ce_pe_option_symbols():
    symbol_oc = fetch_option_chain()

    if symbol_oc != None:
        # Convert the response data into a DataFrame
        oc_df = pd.DataFrame(symbol_oc['data']['optionsChain'])

        # Find the first 'CE' symbol from the top
        first_ce_symbol = None
        for index, row in oc_df.iterrows():
            if row['option_type'] == 'CE':
                first_ce_symbol = row['symbol']
                first_ce_strike = row['strike_price']
                break

        # Find the first 'PE' symbol from the bottom
        first_pe_symbol = None
        for index, row in oc_df[::-1].iterrows():  # Iterate in reverse
            if row['option_type'] == 'PE':
                first_pe_symbol = row['symbol']
                first_pe_strike = row['strike_price']
                break

        return first_ce_symbol, first_pe_symbol, first_ce_strike, first_pe_strike

In [ ]:
def onmessage_ce(ce_message):
    global ce_ltp, index_ltp, unsubscribe_done
    try:
        if ce_message['symbol'] == ce_symbol:
            if "ltp" in ce_message:
                ce_ltp = ce_message["ltp"]
                ce_ltp = float(ce_ltp)

        elif ce_message['symbol'] == index_symbol:
            if "ltp" in ce_message:
                index_ltp = ce_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [ce_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {ce_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessageCE): {e}")


def onerror_ce(message):
    print("CE Error:", message)


def onclose_ce(message):
    print("CE Connection closed:", message)


def onopen_ce():

    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [ce_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def ce_buy_sell_ltp():
    global buy_sell_checked, ce_symbol, ce_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching CE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if ce_symbol is not None and ce_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                ce_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_ce,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_ce,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_ce,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_ce             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                ce_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [ ]:
def onmessage_pe(pe_message):
    global pe_ltp, index_ltp, unsubscribe_done
    try:
        if pe_message['symbol'] == pe_symbol:
            if "ltp" in pe_message:
                pe_ltp = pe_message["ltp"]
                pe_ltp = float(pe_ltp)

        elif pe_message['symbol'] == index_symbol:
            if "ltp" in pe_message:
                index_ltp = pe_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [pe_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {pe_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessagePE): {e}")

def onerror_pe(message):
    print("PE Error:", message)


def onclose_pe(message):
    print("PE Connection closed:", message)


def onopen_pe():
    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [pe_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def pe_buy_sell_ltp():
    global buy_sell_checked, pe_symbol, pe_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching PE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if pe_symbol is not None and pe_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                pe_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_pe,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_pe,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_pe,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_pe             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                pe_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [ ]:
def place_order(symbol):
    try:
        market_order_data = {
            "symbol": symbol,
            "qty": int(quantity),
            "type": 2,  # Market Order
            "side": 1,
            "productType": "INTRADAY",
            "limitPrice": 0,
            "stopPrice": 0,
            "validity": "DAY",
            "disclosedQty": 0,
            "offlineOrder":False
        }

        market_order_entry = fyers.place_order(data=market_order_data)

        if "id" in market_order_entry:
            market_order_id = market_order_entry["id"]
            market_order_message = market_order_entry["message"]
            print(f"{market_order_message}")

    except Exception as e:
        print(f"Error placing orders: {str(e)}")

In [ ]:
def trail_order(symbol, stoploss):
    while True:
        try:
            stoploss = int(stoploss)
            pending_order = fyers.orderbook()

            matching_orders = [order for order in pending_order["orderBook"] if order["status"] == 6]

            modified_orders = 0

            for order in matching_orders:
                if order['symbol'] == symbol:
                    pending_order_id = order['id']
                    pending_order_side = order['side']
                    pending_order_side = int(pending_order_side)

                    if pending_order_side != 1:
                        data = {
                            "id": pending_order_id,
                            "type": 4,
                            "limitPrice": stoploss - 1,
                            "stopPrice": stoploss
                        }

                        modify = fyers.modify_order(data=data)
                        trail_message = modify["message"]
                        print(f"{trail_message}")

                        if trail_message == "Successfully modified order":
                            modified_orders += 1

            # Check if all matching orders are successfully modified
            if modified_orders == len(matching_orders):
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print("Error modifying order:" + str(e))

In [ ]:
def exit_active_order(symbol):
    while True:
        try:
            data = {
                "id":f"{symbol}-INTRADAY"
            }

            exit_response = fyers.exit_positions(data=data)

            if ["message"] in exit_response:
                print(exit_response["message"])
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print(f"Error exiting Order: {e}")

In [ ]:
def reset_flags():
    global active_order, buy_sell_checked

    active_order = False
    buy_sell_checked = False

In [ ]:
# Function to save profits and losses
def save_overall(overall_win, overall_loss, capital):
    trade_type = {
        "overall_win": overall_win,
        "overall_loss": overall_loss,
        "capital": capital
    }

    with open("trade_data.json", "w") as file:
        json.dump(trade_type, file)


# Function to load wins and losses
def load_overall():
    try:
        with open('trade_data.json') as file:
            return json.load(file)
    except FileNotFoundError:
        return None

In [ ]:
def handle_active_ce_order():
    def ce_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, ce_ltp, index_ltp, ce_strike, ce_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        ce_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if ce_ltp != 0 and index_ltp != 0:
                    ce_ltp_array.append(ce_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)


                    if index_ltp <= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(ce_symbol)

                        points = int(ce_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("CE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp >= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                            trailing_sl_inside = int(ce_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp - stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                    else:
                        if (index_ltp - prev_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(ce_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp - trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("CE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=ce_order_loop()).start()

In [ ]:
def handle_active_pe_order():
    def pe_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, pe_ltp, index_ltp, pe_strike, pe_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        pe_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if pe_ltp != 0 and index_ltp != 0:
                    pe_ltp_array.append(pe_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)

                    if index_ltp >= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(pe_symbol)

                        points = int(pe_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("PE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp_array}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp <= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                            trailing_sl_inside = int(pe_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp + stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                    else:
                        if (prev_ltp - index_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(pe_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp + trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("PE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=pe_order_loop()).start()

In [ ]:
def ce_entry():
    threading.Thread(target=ce_buy_sell_ltp).start()

    def ce_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if ce_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = ce_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp + target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp - trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(ce_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_ce_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=ce_entry_thread()).start()

In [ ]:
def pe_entry():
    threading.Thread(target=pe_buy_sell_ltp).start()

    def pe_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if pe_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = pe_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp - target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp + trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(pe_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_pe_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=pe_entry_thread()).start()

In [ ]:
def market_entry_exit_logic(action, actual_closing_price, final_df):
    global sl_hit_condition, unsubscribe_done, ce_ltp, pe_ltp, index_ltp, fixed_ltp, fixed_index_ltp, prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, ce_strike, pe_strike, ce_symbol, pe_symbol

    ce_ltp = 0
    pe_ltp  =0
    index_ltp = 0
    fixed_ltp = 0
    fixed_index_ltp = 0
    prev_ltp = 0
    target_inside = 0
    target_index_inside = 0
    trailing_sl_inside = 0
    trailing_index_inside = 0
    ce_strike = None
    pe_strike = None
    ce_symbol = None
    pe_symbol = None

    final_current_time = final_df.index[-1].time()
    print(final_current_time)

    # Ensure trading occurs only during market hours
    if final_current_time >= dt_time(9, (15 + interval_minutes)) and final_current_time <= dt_time(18, 0):
        if action == 1 and not active_order:  # Buy action
            print("Buy action detected by PPO model.")
            pygame.mixer.music.play()
            sl_hit_condition = False
            unsubscribe_done = False
            print(f"Making CE Position at {actual_closing_price}")
            ce_entry()  # Execute CE entry logic
        elif action == 2 and not active_order:  # Sell action
            print("Sell action detected by PPO model.")
            pygame.mixer.music.play()
            sl_hit_condition = False
            unsubscribe_done = False
            print(f"Making PE Position at {actual_closing_price}")
            pe_entry()  # Execute PE entry logic
        elif action == 0:
            print("Hold action detected by PPO model. No trade executed.")

In [ ]:
def fetch_and_prepare_final_data():
    final_df = fetch_candle_data(10)

    final_data_processor = DataProcessor(final_df, live_processing=True)

    final_processed_df = final_data_processor.process().df

    return final_processed_df

In [ ]:
def find_local_extrema(df):
    order=5
    atr_multiplier=1.5
    min_distance=5

    # Find local maxima and minima
    local_max = argrelextrema(df['high'].values, np.greater_equal, order=order)[0]
    local_min = argrelextrema(df['low'].values, np.less_equal, order=order)[0]

    # Calculate the threshold based on ATR
    threshold = df['atr'] * atr_multiplier

    # Filter by significance
    significant_max = []
    significant_min = []

    for idx in local_max:
        if idx > order and idx < len(df) - order:
            high = df['high'].iloc[idx]
            if significant_min:
                low = df['low'].iloc[significant_min[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_max.append(idx)
            else:
                significant_max.append(idx)

    for idx in local_min:
        if idx > order and idx < len(df) - order:
            low = df['low'].iloc[idx]
            if significant_max:
                high = df['high'].iloc[significant_max[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_min.append(idx)
            else:
                significant_min.append(idx)

    # Ensure minimum distance
    def filter_by_distance(points, min_distance):
        filtered_points = []
        for i in range(len(points)):
            if not filtered_points or (points[i] - filtered_points[-1]) > min_distance:
                filtered_points.append(points[i])
        return filtered_points

    significant_max = filter_by_distance(significant_max, min_distance)
    significant_min = filter_by_distance(significant_min, min_distance)

    return significant_max, significant_min

In [ ]:
def get_trendline(df, point1, point2, kind='high'):
    x = [point1, point2]
    if kind == 'high':
        y = df['high'].values[x]
    else:
        y = df['low'].values[x]
    coeffs = np.polyfit(x, y, 1)
    trendline = np.polyval(coeffs, range(len(df)))
    return trendline

In [ ]:
# Online training configuration
online_learning_steps = 1000  # Steps for each retraining
retrain_interval_minutes = 60  # Retrain the model every 60 minutes

start_time = datetime.now()

while True:
    clear_output(wait=True)

    num_candles = 100

    final_df = fetch_and_prepare_final_data()

    final_df = final_df.iloc[:-1]

    target = final_df['Target'].iloc[-1]
    trailing_sl = final_df['Stop Loss'].iloc[-1]

    # Initialize the TradingEnvironment with the latest data
    live_env = create_env(final_df, config)
    obs = live_env.reset()

    # Use PPO model to predict action for the latest data point
    action, _ = ppo_model.predict(obs, deterministic=True)

    # Extract the actual closing price
    actual_closing_price = final_df['close'].iloc[-1]

    # Retrain PPO model at specified intervals
    if (datetime.now() - start_time).seconds >= retrain_interval_minutes * 60:
        print("Retraining PPO model with online data...")
        train_env = TradingEnvironment(final_df, config)  # Create a new training environment
        ppo_model.set_env(train_env)  # Update the PPO model's environment
        ppo_model.learn(total_timesteps=online_learning_steps)  # Retrain the model
        ppo_model.save(model_save_path)  # Save the updated model
        start_time = datetime.now()

    # Identify most recent high and low points
    recent_highs, recent_lows = find_local_extrema(final_df)

    most_recent_high = recent_highs[-1] if len(recent_highs) > 1 else None
    most_recent_low = recent_lows[-1] if len(recent_lows) > 1 else None

    high_trendline = [np.nan] * len(final_df)
    low_trendline = [np.nan] * len(final_df)

    if most_recent_high is not None:
        previous_high = recent_highs[-2] if len(recent_highs) > 2 else most_recent_high
        high_trendline = get_trendline(final_df, previous_high, most_recent_high, kind='high')

    if most_recent_low is not None:
        previous_low = recent_lows[-2] if len(recent_lows) > 2 else most_recent_low
        low_trendline = get_trendline(final_df, previous_low, most_recent_low, kind='low')

    # Prepare candlestick data for mplfinance
    actual_candles = final_df[-num_candles:].copy()

    # Create a DataFrame for mplfinance
    mpf_df = actual_candles[['open', 'high', 'low', 'close']]

    # Create addplot elements for predicted prices and actual close prices
    ap = [
        mpf.make_addplot(final_df['close'][-num_candles:], color='none', panel=0, secondary_y=False, label=f"Actual Price: {final_df['close'].iloc[-1]}"),
        #mpf.make_addplot(y_pred_ensemble_final_plot, color=(0.95, 0.38, 0.25, 1), panel=0, secondary_y=False, label=f'Predicted Prices ({y_pred_ensemble_final_plot[-1]:.2f})')
    ]

    # Add trendlines to the plot
    if most_recent_high is not None:
        ap.append(mpf.make_addplot(high_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

    if most_recent_low is not None:
        ap.append(mpf.make_addplot(low_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

    fig, axlist = mpf.plot(mpf_df, type='candle', style='binancedark', volume=False, addplot=ap,
                        title=f'Chart', ylabel='Price',
                        figsize=(14, 7), returnfig=True)

    for ax in axlist:
        ax.grid(False)

    axlist[0].legend()
    plt.show()

    # Execute market entry/exit logic
    market_entry_exit_logic(action, actual_closing_price, final_df)

    sleep_time = get_sleep_time(interval_minutes)
    print(f"Sleeping for {sleep_time}")
    time.sleep(sleep_time)

NameError: name 'DataProcessor' is not defined